In [ ]:
from pathlib import Path
import random
from collections import defaultdict
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import pandas as pd
import seaborn as sns
from typing import Any, cast
import pickle
from scipy import stats

from pollen_worker.pollen_utils import get_clients_population_dict
from pollen_worker.placements import (
    _pollen_function,
    get_pollen_models,
    _predict_single_client,
    _linear,
    add_n_batches_column_to_clients_stats_table,
    split_clients_training_table,
    sequential_train_models,
    _jacobian_pollen_function,
    _jacobian_linear,
)

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "bold"

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha

In [ ]:
colors = {
    "Pollen": "#377EB8",
    "Round-Robin": "#FF7F00",
    "Batches": "#4DAF4A",
    "Linear (Parrot)": "#E41A1C",
    "Optimum": "#984EA3",
}
colors_devices = {
    "ngongotaha.cl.cam.ac.uk_cuda:0": "#377EB8",
    "ngongotaha.cl.cam.ac.uk_cuda:1": "#FF7F00",
    "ngongotaha.cl.cam.ac.uk_cuda:2": "#4DAF4A",
    "mauao.cl.cam.ac.uk_cuda:0": "#E41A1C",
}

In [ ]:
batch_sizes = {
    "openimage": 20,
    "google_speech": 20,
    "shakespeare_memory": 10,
    "reddit": 20,
}
datasets_list = ["openimage", "google_speech", "shakespeare_memory", "reddit"]
# datasets_list = ['shakespeare_memory', 'reddit']
datasets_map = {
    "openimage": "IC",
    "google_speech": "SR",
    "shakespeare_memory": "TG",
    "reddit": "MLM",
}

In [ ]:
n_workers = {
    "openimage": {
        "ngongotaha.cl.cam.ac.uk": 4,
        "mauao.cl.cam.ac.uk": 14,
    },
    "google_speech": {
        "ngongotaha.cl.cam.ac.uk": 7,
        "mauao.cl.cam.ac.uk": 21,
    },
    "shakespeare_memory": {
        "ngongotaha.cl.cam.ac.uk": 10,
        "mauao.cl.cam.ac.uk": 33,
    },
    "reddit": {
        "ngongotaha.cl.cam.ac.uk": 3,
        "mauao.cl.cam.ac.uk": 14,
    },
}

In [ ]:
cid_samples_dicts = {}
for dataset in datasets_list:
    print(dataset)
    cid_samples_dicts[dataset] = get_clients_population_dict(
        name=dataset, batch_size=batch_sizes[dataset], seed=1337
    )

In [ ]:
ctt_rr_scale1_paths = {
    "openimage": Path(
        "/nfs-share/ls985/pollen_worker/outputs/2023-10-20/14-24-58/clients_training_stats.parquet"
    ),
    "google_speech": Path(
        "/nfs-share/ls985/pollen_worker/outputs/2023-10-20/11-27-22/clients_training_stats.parquet"
    ),
    "shakespeare_memory": Path(
        "/nfs-share/ls985/pollen_worker/outputs/2023-10-20/10-12-51/clients_training_stats.parquet"
    ),
    "reddit": Path(
        "/nfs-share/ls985/pollen_worker/outputs/2023-10-20/12-44-48/clients_training_stats.parquet"
    ),
}

In [ ]:
ctt_dfs = {}
for dataset in datasets_list:
    df = pq.read_table(ctt_rr_scale1_paths[dataset])
    ctt_dfs[dataset] = df

In [ ]:
# Compute mean and std per dataset per device per num batches to be used in the MC for the real load
ctt_real_distributions = {}
for dataset in datasets_list:
    ctt_real_distributions[dataset] = {}
    for device in colors_devices.keys():
        try:
            ctt_real_distributions[dataset][device] = {}
            node_name = device.split("_")[0]
            gpu_name = device.split("_")[1]
            tmp_df = (
                ctt_dfs[dataset]
                .filter(pc.field("node") == pc.scalar(node_name))
                .filter(pc.field("gpu") == pc.scalar(gpu_name))
            )
            tmp_df = add_n_batches_column_to_clients_stats_table(
                tmp_df, batch_sizes[dataset], cid_samples_dicts[dataset]
            )
            unique_batches = sorted(np.unique(tmp_df.column("n_batches").to_numpy()))
            for b in unique_batches:
                tmp_df_b = tmp_df.filter(pc.field("n_batches") == pc.scalar(b))
                y1 = tmp_df_b.column("end_time").to_numpy()
                y0 = tmp_df_b.column("start_time").to_numpy()
                delta = (y1 - y0) * 1e-9
                ctt_real_distributions[dataset][device][b] = {
                    "median": np.median(delta),
                    "mode": stats.mode(delta)[0],
                    "mean": delta.mean(),
                    "std": delta.std(),
                    "samples": delta,
                }
        except Exception as e:
            print(dataset, device, e)
# ctt_real_distributions

In [ ]:
pollen_models = {}
for dataset in datasets_list:
    pollen_models[dataset], _ = get_pollen_models(
        cid_samples_dicts[dataset], "lb", batch_sizes[dataset], ctt_dfs[dataset]
    )
# print(pollen_models)
linear_models = {}
for dataset in datasets_list:
    linear_models[dataset], _ = get_pollen_models(
        cid_samples_dicts[dataset], "llb", batch_sizes[dataset], ctt_dfs[dataset]
    )
# print(linear_models)

In [ ]:
for dataset in datasets_list:
    plt.figure(figsize=(10, 5))
    x = np.linspace(
        1,
        max([v for k, v in cid_samples_dicts[dataset].items()]) // batch_sizes[dataset],
        250,
    )
    max_y_axis = 0
    for k, v in pollen_models[dataset].items():
        # if 'mauao' in k:
        node_name = k.split("_")[0]
        gpu_name = k.split("_")[1]
        tmp_df = (
            ctt_dfs[dataset]
            .filter(pc.field("node") == pc.scalar(node_name))
            .filter(pc.field("gpu") == pc.scalar(gpu_name))
        )
        tmp_df = add_n_batches_column_to_clients_stats_table(
            tmp_df, batch_sizes[dataset], cid_samples_dicts[dataset]
        )
        x_data = tmp_df.column("n_batches").to_numpy()
        y1_data = tmp_df.column("end_time").to_numpy()
        y0_data = tmp_df.column("start_time").to_numpy()
        plt.scatter(
            x_data,
            (y1_data - y0_data) * 1e-9,
            label=f"Data - {k}",
            alpha=0.3,
            c=colors_devices[k],
            marker=".",
        )
        plt.plot(
            x, _pollen_function(x, *v[0]), label=f"Fit curve - {k}", c=colors_devices[k]
        )
        max_y_axis = max(max_y_axis, max(_pollen_function(x, *v[0])))
    plt.ylim(top=max_y_axis * 1.1)
    plt.ylabel("Time [s]")
    plt.xlabel("Number of batches")
    plt.legend()
    plt.title(f"{dataset.upper()} - clients training time")
    plt.grid()

In [ ]:
for dataset in datasets_list:
    plt.figure(figsize=(10, 5))
    x = np.linspace(
        1,
        max([v for k, v in cid_samples_dicts[dataset].items()]) // batch_sizes[dataset],
        250,
    )
    max_y_axis = 0.0
    for k, v in linear_models[dataset].items():
        node_name = k.split("_")[0]
        gpu_name = k.split("_")[1]
        tmp_df = (
            ctt_dfs[dataset]
            .filter(pc.field("node") == pc.scalar(node_name))
            .filter(pc.field("gpu") == pc.scalar(gpu_name))
        )
        tmp_df = add_n_batches_column_to_clients_stats_table(
            tmp_df, batch_sizes[dataset], cid_samples_dicts[dataset]
        )
        x_data = tmp_df.column("n_batches").to_numpy()
        y1_data = tmp_df.column("end_time").to_numpy()
        y0_data = tmp_df.column("start_time").to_numpy()
        plt.scatter(
            x_data,
            (y1_data - y0_data) * 1e-9,
            label=f"Data - {k}",
            alpha=0.3,
            c=colors_devices[k],
            marker=".",
        )
        plt.plot(x, _linear(x, *v[0]), label=f"Fit curve - {k}", c=colors_devices[k])
        max_y_axis = max(max_y_axis, max(_linear(x, *v[0])))
    plt.ylim(top=max_y_axis * 1.1)
    plt.ylabel("Time [s]")
    plt.xlabel("Number of batches")
    plt.legend()
    plt.title(f"{dataset.upper()} - clients training time")
    plt.grid()

In [ ]:
def create_rr_splits(cids, n_total_workers):
    return np.array_split(cids, n_total_workers)

In [ ]:
def create_batches_splits(cids, cid_samples_dict, n_total_workers, batch_size):
    cids = sorted(
        cids,
        key=lambda x: cid_samples_dict[x],
        reverse=True,
    )
    tmp_splits = [[c] for c in cids[:n_total_workers]]
    for virtual_cid in cids[n_total_workers:]:
        sums = [
            sum([(cid_samples_dict[x] // batch_size) for x in list_cids])
            for list_cids in tmp_splits
        ]
        min_worker = np.argmin(sums)
        tmp_splits[min_worker].append(virtual_cid)
    return [np.array([x for x in list_cids]) for list_cids in tmp_splits]

In [ ]:
def build_pa_tables(
    list_of_devices: list[str],
    list_of_ctt: list[float],
    list_of_n_batches: list[int],
    list_of_server_rounds: list[int],
) -> dict[str, pa.Table]:
    clients_training_stats = pa.Table.from_pydict({
        "node": [device.split("_")[0] for device in list_of_devices],
        "gpu": [device.split("_")[1] for device in list_of_devices],
        "start_time": [0] * len(list_of_devices),
        "end_time": [int(ctt * 1e9) for ctt in list_of_ctt],
        "n_batches": list_of_n_batches,
        "server_round": list_of_server_rounds,
    })
    return split_clients_training_table(clients_training_stats)

In [ ]:
def build_correction_tables(
    tables: dict[str, pa.Table],
    server_round: int,
) -> dict[str, pa.Table]:
    correction_tables: dict[str, pa.Table] = {}
    for client_id, _client_stats in tables.items():
        filtered_client_stats = _client_stats.filter(
            pc.field("server_round") == pc.scalar(server_round - 1),
            null_selection_behavior="emit_null",
        )
        y1 = np.array(filtered_client_stats.column("end_time").flatten())
        y0 = np.array(filtered_client_stats.column("start_time").flatten())
        ctt = (y1 - y0) * 1e-9
        filtered_client_stats = filtered_client_stats.add_column(
            0,
            "ctt",
            cast(pa.Array, pa.array(np.array(ctt).flatten())),
        )
        correction_tables[client_id] = filtered_client_stats.group_by(
            "n_batches"
        ).aggregate([("ctt", "mean")])
    return correction_tables

In [ ]:
def _get_real_load(_dataset, n_batches, device_key):
    # Get real device load
    real_load = -1.0
    while real_load < 0.0:
        try:
            median = ctt_real_distributions[_dataset][device_key][n_batches]["median"]
            mode = ctt_real_distributions[_dataset][device_key][n_batches]["mode"]
            mean = ctt_real_distributions[_dataset][device_key][n_batches]["mean"]
            std = ctt_real_distributions[_dataset][device_key][n_batches]["std"]
            # estimated = _predict_single_client(
            #     # model=_pollen_models[worker[3]+"_"+worker[4]],
            #     model=pollen_models[_dataset][device_key],
            #     fn=_pollen_function,
            #     n_samples=n_batches*batch_sizes[_dataset],
            #     batch_size=batch_sizes[_dataset],
            # )
            # random_sample = random.sample(list(ctt_real_distributions[_dataset][device_key][n_batches]['samples']), 1)[0]
            real_load = random.gauss(mean, std)
            # real_load = mode
            # real_load = random_sample
            # real_load = median
            # real_load = mean
            # real_load = estimated
        except Exception:
            n_batches -= 1
    return real_load

In [ ]:
# Set number of clients per round
n_clients_per_round = 1000
# Set the number of rounds
n_rounds = 100
# Set the number of repetitions
repetitions = 20

In [ ]:
def _get_corrected_load(
    _correction_tables: dict[str, pa.Table] | None, _worker, _n_batches, _load
) -> float:
    if _correction_tables is not None:
        correction = _correction_tables[f"{_worker[3]}_{_worker[4]}"].filter(
            pc.field("n_batches") == pc.scalar(_n_batches)
        )
        if correction.num_rows > 0:
            correction = correction.column("ctt_mean").to_numpy()[0]
            _load = (_load + correction) / 2
    return _load

In [ ]:
# Loop over datasets
simulation_results_corr = {}
# Loop over datasets
for dataset in datasets_list:
    max_real_idle_time = defaultdict(list)
    avg_real_idle_time = defaultdict(list)
    max_estimated_idle_time = defaultdict(list)
    max_batches_difference = defaultdict(list)
    # Loop over placement policies
    for policy in ["Round-Robin", "Batches", "Optimum", "Linear (Parrot)", "Pollen"]:
        # Set global seed
        random.seed(1337)
        # Loop over repetitions
        for rep in tqdm(range(repetitions)):
            # Init lists for collecting runnning stats
            list_of_n_batches = []
            list_of_ctt = []
            list_of_devices = []
            list_of_server_rounds = []
            # Loop over rounds
            for i in range(n_rounds):
                # Sample clients
                if n_clients_per_round > len(cid_samples_dicts[dataset]):
                    # Generate random selection of virtual clients (number of virtual clients per round)
                    cids = random.choices(list(cid_samples_dicts[dataset]), k=n_clients_per_round)  # type: ignore
                else:
                    # Generate random selection of virtual clients (number of virtual clients per round)
                    cids = random.sample(list(cid_samples_dicts[dataset]), k=n_clients_per_round)  # type: ignore
                # Get clients' splits
                n_total_workers = sum(
                    [n_workers[dataset][str(k)[:-7]] for k, _ in colors_devices.items()]
                )
                # Init assignment
                workers_assignments = []
                for k, _ in colors_devices.items():
                    concurrency = n_workers[dataset][str(k)[:-7]]
                    for _ in range(concurrency):
                        # Pollen uses `concurrency` workers per device
                        workers_assignments.append([
                            0.0,  # Real device load
                            0.0,  # Estimated device load
                            0,  # Sum of batches
                            k.split("_")[0],  # Node name
                            k.split("_")[1],  # Device name
                        ])
                if policy == "Round-Robin":
                    splits = create_rr_splits(cids, n_total_workers)
                elif policy == "Batches":
                    splits = create_batches_splits(
                        cids,
                        cid_samples_dicts[dataset],
                        n_total_workers,
                        batch_sizes[dataset],
                    )
                if policy == "Batches" or policy == "Round-Robin":
                    # Assign splits
                    while len(splits) > 0:
                        # Loop over devices
                        for worker in workers_assignments:
                            current_split = splits.pop(0)
                            if len(current_split) > 0:
                                for virtual_client in current_split:
                                    # Get real device load
                                    n_batches = (
                                        cid_samples_dicts[dataset][virtual_client]
                                        // batch_sizes[dataset]
                                    )
                                    real_load = _get_real_load(
                                        dataset, n_batches, worker[3] + "_" + worker[4]
                                    )
                                    worker[0] += real_load
                                    worker[2] += n_batches
                elif policy == "Pollen":
                    # Train models
                    if len(list_of_n_batches) > 0:
                        # Sort clients by number of batches
                        cids = sorted(
                            cids,
                            key=lambda x: cid_samples_dicts[dataset][x]
                            // batch_sizes[dataset],
                            reverse=True,
                        )
                        # Get previous clients' stats
                        tables = build_pa_tables(
                            list_of_devices=list_of_devices,
                            list_of_ctt=list_of_ctt,
                            list_of_n_batches=list_of_n_batches,
                            list_of_server_rounds=list_of_server_rounds,
                        )
                        correction_tables = build_correction_tables(tables, i)
                        # Train models
                        _pollen_models: dict[str, Any] = sequential_train_models(
                            [_pollen_function, _jacobian_pollen_function], tables
                        )
                        # Order models by estimated speed
                        _pollen_models = dict(
                            sorted(
                                _pollen_models.items(),
                                key=lambda item: _predict_single_client(
                                    model=item[1],
                                    fn=_pollen_function,
                                    # n_samples=batch_size**2,
                                    n_samples=3 * batch_sizes[dataset],
                                    batch_size=batch_sizes[dataset],
                                ),
                            )
                        )
                        # Init assignment
                        workers_assignments = []
                        for k, _ in _pollen_models.items():
                            concurrency = n_workers[dataset][str(k)[:-7]]
                            for _ in range(concurrency):
                                # Pollen uses `concurrency` workers per device
                                workers_assignments.append([
                                    0.0,  # Real device load
                                    0.0,  # Estimated device load
                                    0,  # Sum of batches
                                    k.split("_")[0],  # Node name
                                    k.split("_")[1],  # Device name
                                ])
                        # Assign initially at least one client per worker
                        for worker in workers_assignments:
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset,
                                n_batches,
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4],
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_pollen_models[
                                    workers_assignments[0][3]
                                    + "_"
                                    + workers_assignments[0][4]
                                ],
                                # model=pollen_models[dataset][workers_assignments[0][3]+"_"+workers_assignments[0][4]],
                                fn=_pollen_function,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(worker[3] + "_" + worker[4])
                            list_of_server_rounds.append(i)
                            # Update worker load
                            worker[0] += real_load
                            worker[1] += load
                            worker[2] += n_batches
                        # Assing all the rest
                        while len(cids) > 0:
                            # Sort devices by estimated load (increasing order)
                            workers_assignments = sorted(
                                workers_assignments,
                                key=lambda x: x[1],
                            )
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset,
                                n_batches,
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4],
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_pollen_models[
                                    workers_assignments[0][3]
                                    + "_"
                                    + workers_assignments[0][4]
                                ],
                                # model=pollen_models[dataset][workers_assignments[0][3]+"_"+workers_assignments[0][4]],
                                fn=_pollen_function,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4]
                            )
                            list_of_server_rounds.append(i)
                            # Update worker load
                            workers_assignments[0][0] += real_load
                            workers_assignments[0][1] += load
                            workers_assignments[0][2] += n_batches
                    else:
                        splits = create_rr_splits(cids, n_total_workers)
                        # Assign splits
                        while len(splits) > 0:
                            # Loop over devices
                            for worker in workers_assignments:
                                current_split = splits.pop(0)
                                if len(current_split) > 0:
                                    for virtual_cid in current_split:
                                        # Get real device load
                                        n_batches = (
                                            cid_samples_dicts[dataset][virtual_cid]
                                            // batch_sizes[dataset]
                                        )
                                        real_load = _get_real_load(
                                            dataset,
                                            n_batches,
                                            worker[3] + "_" + worker[4],
                                        )
                                        list_of_n_batches.append(n_batches)
                                        list_of_ctt.append(real_load)
                                        list_of_devices.append(
                                            worker[3] + "_" + worker[4]
                                        )
                                        list_of_server_rounds.append(i)
                                        worker[0] += real_load
                                        worker[2] += n_batches
                elif policy == "Linear (Parrot)":
                    # Train models
                    if len(list_of_n_batches) > 0:
                        # Sort clients by number of batches
                        cids = sorted(
                            cids,
                            key=lambda x: cid_samples_dicts[dataset][x]
                            // batch_sizes[dataset],
                            reverse=True,
                        )
                        # Get previous clients' stats
                        tables = build_pa_tables(
                            list_of_devices=list_of_devices,
                            list_of_ctt=list_of_ctt,
                            list_of_n_batches=list_of_n_batches,
                            list_of_server_rounds=list_of_server_rounds,
                        )
                        correction_tables = build_correction_tables(tables, i)
                        # Train models
                        _parrot_models: dict[str, Any] = sequential_train_models(
                            [_linear, _jacobian_linear], tables
                        )
                        # Order models by estimated speed
                        _parrot_models = dict(
                            sorted(
                                _parrot_models.items(),
                                key=lambda item: _predict_single_client(
                                    model=item[1],
                                    fn=_linear,
                                    # n_samples=batch_size**2,
                                    n_samples=3 * batch_sizes[dataset],
                                    batch_size=batch_sizes[dataset],
                                ),
                            )
                        )
                        # Assign initially at least one client per worker
                        for worker in workers_assignments:
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset, n_batches, worker[3] + "_" + worker[4]
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_parrot_models[worker[3] + "_" + worker[4]],
                                # model=linear_models[dataset][worker[3]+"_"+worker[4]],
                                fn=_linear,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(worker[3] + "_" + worker[4])
                            list_of_server_rounds.append(i)
                            # Update worker load
                            worker[0] += real_load
                            worker[1] += load
                            worker[2] += n_batches
                        # Assing all the rest
                        while len(cids) > 0:
                            # Sort devices by estimated load (increasing order)
                            workers_assignments = sorted(
                                workers_assignments,
                                key=lambda x: x[1],
                            )
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset,
                                n_batches,
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4],
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_parrot_models[worker[3] + "_" + worker[4]],
                                # model=linear_models[dataset][worker[3]+"_"+worker[4]],
                                fn=_linear,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4]
                            )
                            list_of_server_rounds.append(i)
                            # Update worker load
                            workers_assignments[0][0] += real_load
                            workers_assignments[0][1] += load
                            workers_assignments[0][2] += n_batches
                    else:
                        splits = create_rr_splits(cids, n_total_workers)
                        # Assign splits
                        while len(splits) > 0:
                            # Loop over devices
                            for worker in workers_assignments:
                                current_split = splits.pop(0)
                                if len(current_split) > 0:
                                    for virtual_cid in current_split:
                                        # Get real device load
                                        n_batches = (
                                            cid_samples_dicts[dataset][virtual_cid]
                                            // batch_sizes[dataset]
                                        )
                                        real_load = _get_real_load(
                                            dataset,
                                            n_batches,
                                            worker[3] + "_" + worker[4],
                                        )
                                        list_of_n_batches.append(n_batches)
                                        list_of_ctt.append(real_load)
                                        list_of_devices.append(
                                            worker[3] + "_" + worker[4]
                                        )
                                        list_of_server_rounds.append(i)
                                        worker[0] += real_load
                                        worker[2] += n_batches
                elif policy == "Optimum":
                    # Assign initially at least one client per worker
                    for worker in workers_assignments:
                        # Extract the first element of the list
                        virtual_cid = cids.pop(0)
                        n_batches = (
                            cid_samples_dicts[dataset][virtual_cid]
                            // batch_sizes[dataset]
                        )
                        # Get real device load
                        real_load = _get_real_load(
                            dataset, n_batches, worker[3] + "_" + worker[4]
                        )
                        # Update worker load
                        worker[0] += real_load
                        worker[2] += n_batches
                    while len(cids) > 0:
                        # Sort devices by real load (increasing order)
                        workers_assignments = sorted(
                            workers_assignments,
                            key=lambda x: x[0],
                        )
                        # Extract the first element of the list
                        virtual_cid = cids.pop(0)
                        n_batches = (
                            cid_samples_dicts[dataset][virtual_cid]
                            // batch_sizes[dataset]
                        )
                        # Get real device load
                        real_load = _get_real_load(
                            dataset,
                            n_batches,
                            workers_assignments[0][3] + "_" + workers_assignments[0][4],
                        )
                        # Update worker load
                        workers_assignments[0][0] += real_load
                        workers_assignments[0][2] += n_batches

                # Compute max load per device
                current_max_real_loads = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_max_real_loads[k] = max(w[0] for w in tmp_worker)
                current_max_real_loads = sorted(
                    current_max_real_loads.items(), key=lambda x: x[1], reverse=True
                )
                current_max_real_loads = [v[1] for v in current_max_real_loads]
                current_avg_real_loads = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_avg_real_loads[k] = np.mean([w[0] for w in tmp_worker])
                current_avg_real_loads = sorted(
                    current_avg_real_loads.items(), key=lambda x: x[1], reverse=True
                )
                current_avg_real_loads = [v[1] for v in current_avg_real_loads]
                current_max_estimated_loads = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_max_estimated_loads[k] = max(w[1] for w in tmp_worker)
                current_max_estimated_loads = sorted(
                    current_max_estimated_loads.items(),
                    key=lambda x: x[1],
                    reverse=True,
                )
                current_max_estimated_loads = [
                    v[1] for v in current_max_estimated_loads
                ]
                current_max_batches = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_max_batches[k] = max(w[2] for w in tmp_worker)
                current_max_batches = sorted(
                    current_max_batches.items(), key=lambda x: x[1], reverse=True
                )
                current_max_batches = [v[1] for v in current_max_batches]
                # Append results
                max_real_idle_time[policy].append(
                    current_max_real_loads[0] - current_max_real_loads[1]
                )
                avg_real_idle_time[policy].append(
                    current_avg_real_loads[0] - current_avg_real_loads[1]
                )
                max_estimated_idle_time[policy].append(
                    current_max_estimated_loads[0] - current_max_estimated_loads[1]
                )
                max_batches_difference[policy].append(
                    current_max_batches[0] - current_max_batches[-1]
                )
    simulation_results_corr[dataset] = {
        "real_idle_time": max_real_idle_time,
        "avg_real_idle_time": avg_real_idle_time,
        "estimated_idle_time": max_estimated_idle_time,
        "batches_difference": max_batches_difference,
    }

In [ ]:
with open(
    f"simulation_results_{n_clients_per_round}_n_clients_per_round_{n_rounds}_rounds_{repetitions}_repetitions_corr_load.pickle",
    "wb",
) as handle:
    pickle.dump(simulation_results_corr, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
f"simulation_results_{n_clients_per_round}_n_clients_per_round_{n_rounds}_rounds_{repetitions}_repetitions_corr_load.pickle"

In [ ]:
with open(
    f"simulation_results_{n_clients_per_round}_n_clients_per_round_{n_rounds}_rounds_{1}_repetitions_corr_load.pickle",
    "rb",
) as handle:
    simulation_results_corr = pickle.load(handle)

In [ ]:
# order_placements = ['Optimum', 'Pollen', 'Round-Robin', 'Batches']
order_placements = ["Pollen", "Round-Robin", "Batches"]
dfs = []
for dataset in datasets_list:
    print(dataset)
    max_real_idle_time = simulation_results_corr[dataset]["real_idle_time"]
    for k, v in max_real_idle_time.items():
        dfs.append(
            pd.DataFrame.from_dict({
                "placement": [k] * len(v),
                "idle_time": v,
                "dataset": [datasets_map[dataset]] * len(v),
                # 'repetition': np.array([[i]*100 for i in range(0, len(max_real_idle_time[k])//1)]).flatten(),
            })
        )
df = pd.concat(dfs, ignore_index=True)
# print(len(df))
# df = df.groupby(['repetition', 'dataset', 'placement']).aggregate('sum').reset_index()
# print(len(df))
textures = ["", ".", "x", "O"]
colors = [
    sns.color_palette("colorblind")[3],
    sns.color_palette("colorblind")[1],
    sns.color_palette("colorblind")[2],
    sns.color_palette("colorblind")[0],
]
alphas = [0.4, 0.6, 0.8, 1.0]
fig, ax = plt.subplots()
sns.barplot(
    x="dataset",
    y="idle_time",
    hue="placement",
    data=df,
    estimator=np.sum,
    ax=ax,
    hue_order=order_placements,
    order=["TG", "IC", "SR", "MLM"],
    edgecolor="black",
)
print(ax.patches)
for i, patch in enumerate(ax.patches):
    if i > 11:
        patch.set_facecolor("w")
        patch.set_hatch(textures[i % len(textures)])
        patch.set_alpha(1.0)
    else:
        patch.set_facecolor(colors[i % len(textures)])
        patch.set_hatch(textures[i // len(textures)])
        patch.set_alpha(alphas[i // len(alphas)])
# plt.yscale('log')
plt.ylabel("Cumulative idle time [s]")
plt.xlabel("")
plt.legend().set_title("")
# plt.title(f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round - {n_rounds} rounds")
plt.grid(axis="y")
# plt.savefig(f"MC_cum_real_idletime_mw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
plt.show()

In [ ]:
for dataset in datasets_list:
    max_real_idle_time = simulation_results_corr[dataset]["real_idle_time"]
    avg_real_idle_time = simulation_results_corr[dataset]["avg_real_idle_time"]
    max_estimated_idle_time = simulation_results_corr[dataset]["estimated_idle_time"]
    max_batches_difference = simulation_results_corr[dataset]["batches_difference"]
    for k, v in max_real_idle_time.items():
        plt.hist(v, bins="auto", label=k, density=True, color=colors[k], alpha=0.6)
    plt.ylabel("Density")
    plt.xlabel("Real Idle time (max worker) [s]")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.legend()
    plt.grid()
    # plt.savefig(f"MC_real_idletime_mw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    dfs = []
    for k, v in max_real_idle_time.items():
        dfs.append(
            pd.DataFrame.from_dict({
                "placement": [k] * len(v),
                "idle_time": v,
            })
        )
    df = pd.concat(dfs)
    sns.barplot(x="placement", y="idle_time", data=df, estimator=np.sum)
    plt.yscale("log")
    plt.ylabel("Cumulative real idle time (max worker) [s]")
    plt.xlabel("")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.grid()
    # plt.savefig(f"MC_cum_real_idletime_mw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    for k, v in avg_real_idle_time.items():
        plt.hist(v, bins="auto", label=k, density=True, color=colors[k], alpha=0.6)
    plt.ylabel("Density")
    plt.xlabel("Real Idle time (avg worker) [s]")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.legend()
    plt.grid()
    # plt.savefig(f"MC_real_idletime_aw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    dfs = []
    for k, v in avg_real_idle_time.items():
        dfs.append(
            pd.DataFrame.from_dict({
                "placement": [k] * len(v),
                "idle_time": v,
            })
        )
    df = pd.concat(dfs)
    sns.barplot(x="placement", y="idle_time", data=df, estimator=np.sum)
    plt.yscale("log")
    plt.ylabel("Cumulative real idle time (avg worker) [s]")
    plt.xlabel("")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.grid()
    # plt.savefig(f"MC_cum_real_idletime_aw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    for k, v in max_batches_difference.items():
        plt.hist(v, bins="auto", label=k, density=True, color=colors[k], alpha=0.6)
    plt.ylabel("Density")
    plt.xlabel("Difference between # batches (max worker - min worker)")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.legend()
    plt.grid()
    # plt.savefig(f"MC_batches_diff_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()

In [ ]:
def _get_corrected_load1(
    _correction_tables: dict[str, pa.Table] | None, _worker, _n_batches, _load
) -> float:
    if _correction_tables is not None:
        correction = _correction_tables[f"{_worker[3]}_{_worker[4]}"].filter(
            pc.field("n_batches") == pc.scalar(_n_batches)
        )
        if correction.num_rows > 0:
            correction = correction.column("ctt_mean").to_numpy()[0]
            _load = correction
    return _load

In [ ]:
# Loop over datasets
simulation_results_corr1 = {}
# Loop over datasets
for dataset in datasets_list:
    max_real_idle_time = defaultdict(list)
    avg_real_idle_time = defaultdict(list)
    max_estimated_idle_time = defaultdict(list)
    max_batches_difference = defaultdict(list)
    # Loop over placement policies
    for policy in ["Round-Robin", "Batches", "Optimum", "Linear (Parrot)", "Pollen"]:
        # Set global seed
        random.seed(1337)
        # Loop over repetitions
        for rep in tqdm(range(repetitions)):
            # Init lists for collecting runnning stats
            list_of_n_batches = []
            list_of_ctt = []
            list_of_devices = []
            list_of_server_rounds = []
            # Loop over rounds
            for i in range(n_rounds):
                # Sample clients
                if n_clients_per_round > len(cid_samples_dicts[dataset]):
                    # Generate random selection of virtual clients (number of virtual clients per round)
                    cids = random.choices(list(cid_samples_dicts[dataset]), k=n_clients_per_round)  # type: ignore
                else:
                    # Generate random selection of virtual clients (number of virtual clients per round)
                    cids = random.sample(list(cid_samples_dicts[dataset]), k=n_clients_per_round)  # type: ignore
                # Get clients' splits
                n_total_workers = sum(
                    [n_workers[dataset][str(k)[:-7]] for k, _ in colors_devices.items()]
                )
                # Init assignment
                workers_assignments = []
                for k, _ in colors_devices.items():
                    concurrency = n_workers[dataset][str(k)[:-7]]
                    for _ in range(concurrency):
                        # Pollen uses `concurrency` workers per device
                        workers_assignments.append([
                            0.0,  # Real device load
                            0.0,  # Estimated device load
                            0,  # Sum of batches
                            k.split("_")[0],  # Node name
                            k.split("_")[1],  # Device name
                        ])
                if policy == "Round-Robin":
                    splits = create_rr_splits(cids, n_total_workers)
                elif policy == "Batches":
                    splits = create_batches_splits(
                        cids,
                        cid_samples_dicts[dataset],
                        n_total_workers,
                        batch_sizes[dataset],
                    )
                if policy == "Batches" or policy == "Round-Robin":
                    # Assign splits
                    while len(splits) > 0:
                        # Loop over devices
                        for worker in workers_assignments:
                            current_split = splits.pop(0)
                            if len(current_split) > 0:
                                for virtual_client in current_split:
                                    # Get real device load
                                    n_batches = (
                                        cid_samples_dicts[dataset][virtual_client]
                                        // batch_sizes[dataset]
                                    )
                                    real_load = _get_real_load(
                                        dataset, n_batches, worker[3] + "_" + worker[4]
                                    )
                                    worker[0] += real_load
                                    worker[2] += n_batches
                elif policy == "Pollen":
                    # Train models
                    if len(list_of_n_batches) > 0:
                        # Sort clients by number of batches
                        cids = sorted(
                            cids,
                            key=lambda x: cid_samples_dicts[dataset][x]
                            // batch_sizes[dataset],
                            reverse=True,
                        )
                        # Get previous clients' stats
                        tables = build_pa_tables(
                            list_of_devices=list_of_devices,
                            list_of_ctt=list_of_ctt,
                            list_of_n_batches=list_of_n_batches,
                            list_of_server_rounds=list_of_server_rounds,
                        )
                        correction_tables = build_correction_tables(tables, i)
                        # Train models
                        _pollen_models: dict[str, Any] = sequential_train_models(
                            [_pollen_function, _jacobian_pollen_function], tables
                        )
                        # Order models by estimated speed
                        _pollen_models = dict(
                            sorted(
                                _pollen_models.items(),
                                key=lambda item: _predict_single_client(
                                    model=item[1],
                                    fn=_pollen_function,
                                    # n_samples=batch_size**2,
                                    n_samples=3 * batch_sizes[dataset],
                                    batch_size=batch_sizes[dataset],
                                ),
                            )
                        )
                        # Init assignment
                        workers_assignments = []
                        for k, _ in _pollen_models.items():
                            concurrency = n_workers[dataset][str(k)[:-7]]
                            for _ in range(concurrency):
                                # Pollen uses `concurrency` workers per device
                                workers_assignments.append([
                                    0.0,  # Real device load
                                    0.0,  # Estimated device load
                                    0,  # Sum of batches
                                    k.split("_")[0],  # Node name
                                    k.split("_")[1],  # Device name
                                ])
                        # Assign initially at least one client per worker
                        for worker in workers_assignments:
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset,
                                n_batches,
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4],
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_pollen_models[
                                    workers_assignments[0][3]
                                    + "_"
                                    + workers_assignments[0][4]
                                ],
                                # model=pollen_models[dataset][workers_assignments[0][3]+"_"+workers_assignments[0][4]],
                                fn=_pollen_function,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load1(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(worker[3] + "_" + worker[4])
                            list_of_server_rounds.append(i)
                            # Update worker load
                            worker[0] += real_load
                            worker[1] += load
                            worker[2] += n_batches
                        # Assing all the rest
                        while len(cids) > 0:
                            # Sort devices by estimated load (increasing order)
                            workers_assignments = sorted(
                                workers_assignments,
                                key=lambda x: x[1],
                            )
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset,
                                n_batches,
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4],
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_pollen_models[
                                    workers_assignments[0][3]
                                    + "_"
                                    + workers_assignments[0][4]
                                ],
                                # model=pollen_models[dataset][workers_assignments[0][3]+"_"+workers_assignments[0][4]],
                                fn=_pollen_function,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load1(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4]
                            )
                            list_of_server_rounds.append(i)
                            # Update worker load
                            workers_assignments[0][0] += real_load
                            workers_assignments[0][1] += load
                            workers_assignments[0][2] += n_batches
                    else:
                        splits = create_rr_splits(cids, n_total_workers)
                        # Assign splits
                        while len(splits) > 0:
                            # Loop over devices
                            for worker in workers_assignments:
                                current_split = splits.pop(0)
                                if len(current_split) > 0:
                                    for virtual_cid in current_split:
                                        # Get real device load
                                        n_batches = (
                                            cid_samples_dicts[dataset][virtual_cid]
                                            // batch_sizes[dataset]
                                        )
                                        real_load = _get_real_load(
                                            dataset,
                                            n_batches,
                                            worker[3] + "_" + worker[4],
                                        )
                                        list_of_n_batches.append(n_batches)
                                        list_of_ctt.append(real_load)
                                        list_of_devices.append(
                                            worker[3] + "_" + worker[4]
                                        )
                                        list_of_server_rounds.append(i)
                                        worker[0] += real_load
                                        worker[2] += n_batches
                elif policy == "Linear (Parrot)":
                    # Train models
                    if len(list_of_n_batches) > 0:
                        # Sort clients by number of batches
                        cids = sorted(
                            cids,
                            key=lambda x: cid_samples_dicts[dataset][x]
                            // batch_sizes[dataset],
                            reverse=True,
                        )
                        # Get previous clients' stats
                        tables = build_pa_tables(
                            list_of_devices=list_of_devices,
                            list_of_ctt=list_of_ctt,
                            list_of_n_batches=list_of_n_batches,
                            list_of_server_rounds=list_of_server_rounds,
                        )
                        correction_tables = build_correction_tables(tables, i)
                        # Train models
                        _parrot_models: dict[str, Any] = sequential_train_models(
                            [_linear, _jacobian_linear], tables
                        )
                        # Order models by estimated speed
                        _parrot_models = dict(
                            sorted(
                                _parrot_models.items(),
                                key=lambda item: _predict_single_client(
                                    model=item[1],
                                    fn=_linear,
                                    # n_samples=batch_size**2,
                                    n_samples=3 * batch_sizes[dataset],
                                    batch_size=batch_sizes[dataset],
                                ),
                            )
                        )
                        # Assign initially at least one client per worker
                        for worker in workers_assignments:
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset, n_batches, worker[3] + "_" + worker[4]
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_parrot_models[worker[3] + "_" + worker[4]],
                                # model=linear_models[dataset][worker[3]+"_"+worker[4]],
                                fn=_linear,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load1(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(worker[3] + "_" + worker[4])
                            list_of_server_rounds.append(i)
                            # Update worker load
                            worker[0] += real_load
                            worker[1] += load
                            worker[2] += n_batches
                        # Assing all the rest
                        while len(cids) > 0:
                            # Sort devices by estimated load (increasing order)
                            workers_assignments = sorted(
                                workers_assignments,
                                key=lambda x: x[1],
                            )
                            # Extract the first element of the list
                            virtual_cid = cids.pop(0)
                            num_samples = cid_samples_dicts[dataset][virtual_cid]
                            n_batches = num_samples // batch_sizes[dataset]
                            # Get real device load
                            real_load = _get_real_load(
                                dataset,
                                n_batches,
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4],
                            )
                            # Get estimated device load
                            load = _predict_single_client(
                                model=_parrot_models[worker[3] + "_" + worker[4]],
                                # model=linear_models[dataset][worker[3]+"_"+worker[4]],
                                fn=_linear,
                                n_samples=num_samples,
                                batch_size=batch_sizes[dataset],
                            )
                            load = _get_corrected_load1(
                                correction_tables, worker, n_batches, load
                            )
                            list_of_n_batches.append(n_batches)
                            list_of_ctt.append(real_load)
                            list_of_devices.append(
                                workers_assignments[0][3]
                                + "_"
                                + workers_assignments[0][4]
                            )
                            list_of_server_rounds.append(i)
                            # Update worker load
                            workers_assignments[0][0] += real_load
                            workers_assignments[0][1] += load
                            workers_assignments[0][2] += n_batches
                    else:
                        splits = create_rr_splits(cids, n_total_workers)
                        # Assign splits
                        while len(splits) > 0:
                            # Loop over devices
                            for worker in workers_assignments:
                                current_split = splits.pop(0)
                                if len(current_split) > 0:
                                    for virtual_cid in current_split:
                                        # Get real device load
                                        n_batches = (
                                            cid_samples_dicts[dataset][virtual_cid]
                                            // batch_sizes[dataset]
                                        )
                                        real_load = _get_real_load(
                                            dataset,
                                            n_batches,
                                            worker[3] + "_" + worker[4],
                                        )
                                        list_of_n_batches.append(n_batches)
                                        list_of_ctt.append(real_load)
                                        list_of_devices.append(
                                            worker[3] + "_" + worker[4]
                                        )
                                        list_of_server_rounds.append(i)
                                        worker[0] += real_load
                                        worker[2] += n_batches
                elif policy == "Optimum":
                    # Assign initially at least one client per worker
                    for worker in workers_assignments:
                        # Extract the first element of the list
                        virtual_cid = cids.pop(0)
                        n_batches = (
                            cid_samples_dicts[dataset][virtual_cid]
                            // batch_sizes[dataset]
                        )
                        # Get real device load
                        real_load = _get_real_load(
                            dataset, n_batches, worker[3] + "_" + worker[4]
                        )
                        # Update worker load
                        worker[0] += real_load
                        worker[2] += n_batches
                    while len(cids) > 0:
                        # Sort devices by real load (increasing order)
                        workers_assignments = sorted(
                            workers_assignments,
                            key=lambda x: x[0],
                        )
                        # Extract the first element of the list
                        virtual_cid = cids.pop(0)
                        n_batches = (
                            cid_samples_dicts[dataset][virtual_cid]
                            // batch_sizes[dataset]
                        )
                        # Get real device load
                        real_load = _get_real_load(
                            dataset,
                            n_batches,
                            workers_assignments[0][3] + "_" + workers_assignments[0][4],
                        )
                        # Update worker load
                        workers_assignments[0][0] += real_load
                        workers_assignments[0][2] += n_batches

                # Compute max load per device
                current_max_real_loads = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_max_real_loads[k] = max(w[0] for w in tmp_worker)
                current_max_real_loads = sorted(
                    current_max_real_loads.items(), key=lambda x: x[1], reverse=True
                )
                current_max_real_loads = [v[1] for v in current_max_real_loads]
                current_avg_real_loads = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_avg_real_loads[k] = np.mean([w[0] for w in tmp_worker])
                current_avg_real_loads = sorted(
                    current_avg_real_loads.items(), key=lambda x: x[1], reverse=True
                )
                current_avg_real_loads = [v[1] for v in current_avg_real_loads]
                current_max_estimated_loads = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_max_estimated_loads[k] = max(w[1] for w in tmp_worker)
                current_max_estimated_loads = sorted(
                    current_max_estimated_loads.items(),
                    key=lambda x: x[1],
                    reverse=True,
                )
                current_max_estimated_loads = [
                    v[1] for v in current_max_estimated_loads
                ]
                current_max_batches = {}
                for k, _ in colors_devices.items():
                    tmp_worker = [
                        w for w in workers_assignments if w[3] + "_" + w[4] == k
                    ]
                    current_max_batches[k] = max(w[2] for w in tmp_worker)
                current_max_batches = sorted(
                    current_max_batches.items(), key=lambda x: x[1], reverse=True
                )
                current_max_batches = [v[1] for v in current_max_batches]
                # Append results
                max_real_idle_time[policy].append(
                    current_max_real_loads[0] - current_max_real_loads[1]
                )
                avg_real_idle_time[policy].append(
                    current_avg_real_loads[0] - current_avg_real_loads[1]
                )
                max_estimated_idle_time[policy].append(
                    current_max_estimated_loads[0] - current_max_estimated_loads[1]
                )
                max_batches_difference[policy].append(
                    current_max_batches[0] - current_max_batches[-1]
                )
    simulation_results_corr1[dataset] = {
        "real_idle_time": max_real_idle_time,
        "avg_real_idle_time": avg_real_idle_time,
        "estimated_idle_time": max_estimated_idle_time,
        "batches_difference": max_batches_difference,
    }

In [ ]:
with open(
    f"simulation_results_{n_clients_per_round}_n_clients_per_round_{n_rounds}_rounds_{repetitions}_repetitions_corr_load1.pickle",
    "wb",
) as handle:
    pickle.dump(simulation_results_corr1, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open(
    f"simulation_results_{n_clients_per_round}_n_clients_per_round_{n_rounds}_rounds_{repetitions}_repetitions_corr_load1.pickle",
    "rb",
) as handle:
    pickle.dump(simulation_results_corr1, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
for dataset in datasets_list:
    max_real_idle_time = simulation_results_corr1[dataset]["real_idle_time"]
    avg_real_idle_time = simulation_results_corr1[dataset]["avg_real_idle_time"]
    max_estimated_idle_time = simulation_results_corr1[dataset]["estimated_idle_time"]
    max_batches_difference = simulation_results_corr1[dataset]["batches_difference"]
    for k, v in max_real_idle_time.items():
        plt.hist(v, bins="auto", label=k, density=True, color=colors[k], alpha=0.6)
    plt.ylabel("Density")
    plt.xlabel("Real Idle time (max worker) [s]")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.legend()
    plt.grid()
    # plt.savefig(f"MC_real_idletime_mw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    dfs = []
    for k, v in max_real_idle_time.items():
        dfs.append(
            pd.DataFrame.from_dict({
                "placement": [k] * len(v),
                "idle_time": v,
            })
        )
    df = pd.concat(dfs)
    sns.barplot(x="placement", y="idle_time", data=df, estimator=np.sum)
    plt.yscale("log")
    plt.ylabel("Cumulative real idle time (max worker) [s]")
    plt.xlabel("")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.grid()
    # plt.savefig(f"MC_cum_real_idletime_mw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    for k, v in avg_real_idle_time.items():
        plt.hist(v, bins="auto", label=k, density=True, color=colors[k], alpha=0.6)
    plt.ylabel("Density")
    plt.xlabel("Real Idle time (avg worker) [s]")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.legend()
    plt.grid()
    # plt.savefig(f"MC_real_idletime_aw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    dfs = []
    for k, v in avg_real_idle_time.items():
        dfs.append(
            pd.DataFrame.from_dict({
                "placement": [k] * len(v),
                "idle_time": v,
            })
        )
    df = pd.concat(dfs)
    sns.barplot(x="placement", y="idle_time", data=df, estimator=np.sum)
    plt.yscale("log")
    plt.ylabel("Cumulative real idle time (avg worker) [s]")
    plt.xlabel("")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.grid()
    # plt.savefig(f"MC_cum_real_idletime_aw_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()
    for k, v in max_batches_difference.items():
        plt.hist(v, bins="auto", label=k, density=True, color=colors[k], alpha=0.6)
    plt.ylabel("Density")
    plt.xlabel("Difference between # batches (max worker - min worker)")
    plt.title(
        f"{dataset.upper()} - MC placement - {n_clients_per_round} clients per round -"
        f" {n_rounds} rounds"
    )
    plt.legend()
    plt.grid()
    # plt.savefig(f"MC_batches_diff_{dataset}_{n_clients_per_round}_{n_rounds}_{repetitions}.pdf", format="pdf", dpi=800, bbox_inches='tight')
    plt.show()